# DS2002 · Cleaning Clinic

**Studio — 2026-09-23 · Fall 2026**  
**Class time:** 45 minutes

---

## Write the pipeline, then defend it

Monday I made the cleaning decisions and told you what they were. Today you make them, and the output is two things: a clean frame, and a **decision log** that says what you did to whose rows and why.

The log is not paperwork. On the midterm your team will disagree about whether a refund counts, and the log is what turns that into a two-minute conversation instead of an afternoon of re-deriving numbers.

Every step below follows the same three-part shape: **do it, count what you changed, log the decision.**

In [19]:
import pandas as pd, numpy as np
from io import StringIO
raw = '''order_id,item,category,qty,price,ts
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
1,Cheeseburger,Food,2,$7.50,2026-09-05T12:03:00
2,cheese burger,food,1,7.5,09/05/2026 12:40
3,Foam Finger,Merch,NULL,12,2026-09-05 13:00:00
4,UVA T-Shirt ,Apparel,2,$24.00,2026-09-05 13:05
5,Rain Poncho,RainGear,-3,6,2026-09-05T13:20:00
6,rain poncho,rain-gear,4,$6.00,
7,,Merch,1,12,2026-09-05T14:00:00'''
df = pd.read_csv(StringIO(raw))
df

,order_id,item,category,qty,price,ts
0,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
1,1,Cheeseburger,Food,2.0,$7.50,2026-09-05T12:03:00
2,2,cheese burger,food,1.0,7.5,09/05/2026 12:40
3,3,Foam Finger,Merch,NaN,12,2026-09-05 13:00:00
4,4,UVA T-Shirt,Apparel,2.0,$24.00,2026-09-05 13:05
5,5,Rain Poncho,RainGear,-3.0,6,2026-09-05T13:20:00
6,6,rain poncho,rain-gear,4.0,$6.00,NaN
7,7,NaN,Merch,1.0,12,2026-09-05T14:00:00


### Set up the log

Run this first. Each step calls `log()` with what happened and how many rows it touched.

In [20]:
DECISIONS = []

def log(step, decision, rows_affected):
    DECISIONS.append({'step': step, 'decision': decision, 'rows': rows_affected})
    print(f'[{step}] {decision} ({rows_affected} row(s))')

def show_log():
    return pd.DataFrame(DECISIONS)

raw_rows = len(df)
print('starting with', raw_rows, 'rows')

starting with 8 rows


### Step 0 — take inventory

**TODO:** print the shape, the dtypes, the null count per column, and the number of exact duplicate rows. Do not skip this — the rest of the studio depends on knowing what you have.

In [21]:
print('DataFrame Shape:', df.shape)
print('\nDataFrame Info:')
df.info()
print('\nNull values per column:')
print(df.isnull().sum())
print('\nNumber of exact duplicate rows:', df.duplicated().sum())

DataFrame Shape: (8, 6)

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   order_id  8 non-null      int64  
 1   item      7 non-null      object 
 2   category  8 non-null      object 
 3   qty       7 non-null      float64
 4   price     8 non-null      object 
 5   ts        7 non-null      object 
dtypes: float64(1), int64(1), object(4)
memory usage: 516.0+ bytes

Null values per column:
order_id    0
item        1
category    0
qty         1
price       0
ts          1
dtype: int64

Number of exact duplicate rows: 1


**What is wrong with this data?** List at least five specific problems:

1.  item column: Contains one missing value.
2.  qty column: Contains one missing value and at least one negative value, which might require specific handling (ex. for refunds).
3.  price column: Stored as an object (string) and contains non-numeric characters (like '$'), preventing direct numerical operations. It should be a float.
4.  timestamp column: Stored as an object (string), has inconsistent date/time formats, and contains one missing value. It should be a datetime object.
5.  Duplicate Rows: There is one exact duplicate row in the dataset.

### Step 1 — duplicates

**TODO:** drop exact duplicate rows into a new frame called `clean`, then log how many you removed. Use `.copy()` so later assignments do not warn.

In [22]:
removed = df.duplicated().sum()
clean = df.drop_duplicates().copy()

log('duplicates', 'dropped exact duplicate rows', removed)

[duplicates] dropped exact duplicate rows (1 row(s))


### Step 2 — price into a real number

**TODO:** strip the dollar signs and any stray whitespace, then convert to float. Assert the dtype afterward so you find out now if a stray character survived.

In [23]:
clean['price'] = clean['price'].astype(str).str.replace('$', '', regex=False).str.strip().astype(float)
assert clean['price'].dtype == float
log('price', 'converted price to float and removed currency symbols', raw_rows - len(clean))

[price] converted price to float and removed currency symbols (1 row(s))


### Step 3 — quantity, and two decisions

**TODO:** coerce `qty` to numeric. Then decide, separately:

- what to do with the row that has no quantity
- what to do with the refund (negative quantity)

Log each decision with its row count. There is no single right answer — there is only an answer you can defend.

In [24]:
clean['qty'] = pd.to_numeric(clean['qty'], errors='coerce')

negative_qty_count = (clean['qty'] < 0).sum()
clean.loc[clean['qty'] < 0, 'qty'] = np.nan
log('qty_negative', 'replaced negative quantities with NaN', negative_qty_count)

missing_qty_count = clean['qty'].isnull().sum()
clean['qty'] = clean['qty'].fillna(1)
log('qty_missing', 'filled missing quantities with 1', missing_qty_count)

[qty_negative] replaced negative quantities with NaN (1 row(s))
[qty_missing] filled missing quantities with 1 (2 row(s))


### Step 4 — categories that mean one thing

**TODO:** normalize case and punctuation, then map the remaining variants with an explicit dict. Print the unique values before and after so the collapse is visible. Log how many distinct categories you started and ended with.

In [25]:
print('before:', sorted(clean['category'].unique()))

original_categories_count = clean['category'].nunique()
clean['category'] = clean['category'].astype(str).str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True).str.strip()

CATEGORY_MAP = {
    'food': 'Food',
    'raingear': 'RainGear'
}

clean['category'] = clean['category'].replace(CATEGORY_MAP)

print('after: ', sorted(clean['category'].unique()))
new_categories_count = clean['category'].nunique()
log('category', 'standardized category names', original_categories_count - new_categories_count)

before: ['Apparel', 'Food', 'Merch', 'RainGear', 'food', 'rain-gear']
after:  ['Food', 'RainGear', 'apparel', 'merch']
[category] standardized category names (2 row(s))


### Step 5 — item names

**TODO:** same treatment for `item`. One product is spelled two ways, and one row has no item at all — decide what to do with it.

In [26]:
print('before:', sorted(clean['item'].astype(str).unique()))

original_items_count = clean['item'].nunique()
clean['item'] = clean['item'].astype(str).str.lower().str.replace(r'[^a-z0-9\s]', '', regex=True).str.strip()

missing_items = clean['item'].isnull().sum()
clean['item'] = clean['item'].fillna('unknown item')
log('item_missing', 'filled missing item names with "unknown item"', missing_items)

ITEM_MAP = {
    'cheese burger': 'Cheeseburger',
    'cheeseburger': 'Cheeseburger',
    'foam finger': 'Foam Finger',
    'uva tshirt': 'UVA T-Shirt',
    'rain poncho': 'Rain Poncho',
    'unknown item': 'Unknown Item',
    'nan': 'Unknown Item'
}

clean['item'] = clean['item'].replace(ITEM_MAP)

print('after: ', sorted(clean['item'].unique()))
new_items_count = clean['item'].nunique()
log('item', 'standardized item names', original_items_count - new_items_count)


before: ['Cheeseburger', 'Foam Finger', 'Rain Poncho', 'UVA T-Shirt ', 'cheese burger', 'nan', 'rain poncho']
[item_missing] filled missing item names with "unknown item" (0 row(s))
after:  ['Cheeseburger', 'Foam Finger', 'Rain Poncho', 'UVA T-Shirt', 'Unknown Item']
[item] standardized item names (1 row(s))


### Step 6 — timestamps

**TODO:** parse `ts` into real datetimes, coercing failures to `NaT`. Report how many failed. Then add an `hour` column, which is only possible once the column is a real datetime.

In [27]:
original_ts_na_count = clean['ts'].isnull().sum()
clean['ts'] = pd.to_datetime(clean['ts'], errors='coerce')
failed_ts_count = clean['ts'].isnull().sum() - original_ts_na_count
log('ts', 'parsed timestamps and coerced unparseable values to NaT', failed_ts_count)

if pd.api.types.is_datetime64_any_dtype(clean['ts']):
    clean['hour'] = clean['ts'].dt.hour
    log('ts_hour', 'added hour column from timestamp', clean['ts'].notnull().sum())
else:
    log('ts_hour', 'could not add hour column, ts column is not datetime', 0)

print("Cleaned 'ts' column info:")
clean['ts'].info()
print("\n'hour' column info:")
clean['hour'].info()

[ts] parsed timestamps and coerced unparseable values to NaT (3 row(s))
[ts_hour] added hour column from timestamp (3 row(s))
Cleaned 'ts' column info:
<class 'pandas.core.series.Series'>
Index: 7 entries, 0 to 7
Series name: ts
Non-Null Count  Dtype         
--------------  -----         
3 non-null      datetime64[ns]
dtypes: datetime64[ns](1)
memory usage: 112.0 bytes

'hour' column info:
<class 'pandas.core.series.Series'>
Index: 7 entries, 0 to 7
Series name: hour
Non-Null Count  Dtype  
--------------  -----  
3 non-null      float64
dtypes: float64(1)
memory usage: 112.0 bytes


# Assertions to catch regressions in the pipeline


In [28]:
assert clean['price'].dtype == float, "Price column is not float!"
assert clean['qty'].dtype == float or clean['qty'].dtype == np.float64, "Quantity column is not numeric!"
assert clean['qty'].isnull().sum() == 0, "Incorrect number of NaN values in qty column after cleaning!"
assert (clean['qty'] >= 0).all(skipna=True), "Negative quantities still exist!"
assert clean['item'].isnull().sum() == 0, "Item column still has nulls!"
assert clean['category'].isnull().sum() == 0, "Category column still has nulls!"
assert clean['ts'].dtype == 'datetime64[ns]', "Timestamp column is not datetime!"

print("All assertions passed! Data cleaning seems robust.")

clean['revenue'] = clean['qty'] * clean['price']

total_rows = len(clean)
total_units = clean['qty'].sum()
total_revenue = clean['revenue'].sum()
distinct_categories = clean['category'].nunique()

print(f"\nTotal rows: {total_rows}")
print(f"Total units: {total_units}")
print(f"Total revenue: {total_revenue:.2f}")
print(f"Distinct categories: {distinct_categories}")

All assertions passed! Data cleaning seems robust.

Total rows: 7
Total units: 12.0
Total revenue: 124.50
Distinct categories: 4


### Step 8 — the decision log

**TODO:** print your log. Then answer, in the markdown cell below: which single decision moved your revenue total the most, and what is the number both ways?

In [29]:
display(show_log())

,step,decision,rows
0,duplicates,dropped exact duplicate rows,1
1,price,converted price to float and removed currency ...,1
2,qty_negative,replaced negative quantities with NaN,1
3,qty_missing,filled missing quantities with 1,2
4,category,standardized category names,2
5,item_missing,"filled missing item names with ""unknown item""",0
6,item,standardized item names,1
7,ts,parsed timestamps and coerced unparseable valu...,3
8,ts_hour,added hour column from timestamp,3


**The decision that mattered most:** qty_negative (converting -3 to 1)

**Revenue with it:** $124.50  **Revenue without it:** $100.50

---

## Checkpoint (participation)

Report your row count and revenue after cleaning, and the one decision that moved the total most.

Work in groups if you wish, then fill in the cell below **yourself**. Paste the printed output (or a screenshot of it) into this week's **Studio Checkpoint** in Canvas by **Thursday 11:59pm ET**. One submission per person, not per group.

In [30]:
rows_after = 7
revenue_after = 124.50
biggest_decision = 'qty_negative (converting -3 to 1)'
revenue_other_way = 100.50

print('rows after cleaning:', rows_after)
print('revenue:', revenue_after)
print('decision that mattered:', biggest_decision)
print('revenue the other way:', revenue_other_way)


rows after cleaning: 7
revenue: 124.5
decision that mattered: qty_negative (converting -3 to 1)
revenue the other way: 100.5
